In [5]:
import pandas as pd
import numpy as np

# 1. Load data
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00373/drug_consumption.data"
cols = ["ID", "Age", "Gender", "Education", "Country", "Ethnicity",
        "N", "E", "O", "A", "C", "Impulsive", "SS",
        "Am", "Amyl", "Benzos", "Caff", "Cannabis", "Choc", "Coke",
        "Crack", "Ecstasy", "Heroin", "Ketamine", "LegalH", "LSD",
        "Methadone", "Mushrooms", "Nicotine", "Semer", "VSA"]
df = pd.read_csv(url, header=None, names=cols)

# 2. Select predictors
predictors = ["Age", "Gender", "N", "E", "O", "A", "C", "SS"]

# 3. Map 'CLx' → numerical value
def map_cl_to_num(x):
    if isinstance(x, str) and x.startswith('CL'):
        return int(x[2:])
    else:
        return np.nan

df[predictors] = df[predictors].map(map_cl_to_num)

# 4. Convert to float
X = df[predictors].astype(float)

# 5. Convert response drugs to binary
drugs = ["Am", "Benzos", "Cannabis", "Coke", "Ecstasy", "Ketamine",
         "LegalH", "LSD", "Methadone", "Mushrooms", "Nicotine"]
Y = df[drugs].apply(lambda col: col.map(lambda x: 1 if str(x) in ['CL1','CL2','CL3','CL4','CL5','CL6'] else 0))

# 6. Standardize predictors
X = (X - X.mean()) / X.std()

# 7. Drop any missing rows
X, Y = X.dropna(), Y.loc[X.index]
print("Predictor shape:", X.shape, " | Response shape:", Y.shape)


Predictor shape: (0, 8)  | Response shape: (1885, 11)


In [6]:
# 1. Load SHARE data (simulate structure for now)
# In practice, you’ll use: pd.read_csv("SHARE_wave1.csv")
N = 29207
np.random.seed(42)
age = np.random.normal(65, 10, N)
gender = np.random.choice([0, 1], size=N)  # 0=male, 1=female

# 11 country dummies (ref = Netherlands)
countries = ["AT", "DE", "SE", "ES", "IT", "FR", "DK", "EL", "CH", "BE", "IL"]
country_df = pd.DataFrame(np.eye(len(countries))[np.random.randint(0, len(countries), N)],
                          columns=countries)

# Combine predictors
X_share = pd.DataFrame({"Age": age, "Gender": gender}).join(country_df)

# 2. Simulate binary responses (7 diseases)
diseases = ["Di", "H", "CL", "JD", "An", "S", "De"]
Y_share = pd.DataFrame({
    d: (np.random.rand(N) < np.random.uniform(0.05, 0.35)) * 1 for d in diseases
})

# 3. Standardize predictors
X_share = (X_share - X_share.mean()) / X_share.std()

print("Predictor shape:", X_share.shape, "| Response shape:", Y_share.shape)


Predictor shape: (29207, 13) | Response shape: (29207, 7)


In [8]:
import numpy as np

def compute_deviance(X, Y, B, V, m):
    """
    Computes deviance for the logistic reduced-rank regression model.
    """
    eta = X @ (B @ V.T) + np.ones((X.shape[0], 1)) @ m.reshape(1, -1)
    pi = 1 / (1 + np.exp(-eta))
    
    # Clip to avoid log(0)
    pi = np.clip(pi, 1e-8, 1 - 1e-8)
    logL = np.sum(Y * np.log(pi) + (1 - Y) * np.log(1 - pi))
    dev = -2 * logL
    return dev, pi


In [9]:
from sklearn.linear_model import LogisticRegression

def compute_Qr(X, Y, B, V, m):
    N, R = Y.shape
    Qr_values = []
    D_rankS, _ = compute_deviance(X, Y, B, V, m)
    
    for r in range(R):
        y = Y[:, r]
        
        # 1️⃣ Intercept-only model
        p0 = np.clip(y.mean(), 1e-8, 1 - 1e-8)
        logL0 = np.sum(y * np.log(p0) + (1 - y) * np.log(1 - p0))
        D0 = -2 * logL0
        
        # 2️⃣ Separate full logistic regression
        lr = LogisticRegression(max_iter=500)
        lr.fit(X, y)
        logL_full = lr.score(X, y)  # score = mean accuracy, not log-likelihood
        pi_full = lr.predict_proba(X)[:, 1]
        pi_full = np.clip(pi_full, 1e-8, 1 - 1e-8)
        logL_full = np.sum(y * np.log(pi_full) + (1 - y) * np.log(1 - pi_full))
        D_full = -2 * logL_full
        
        # 3️⃣ Deviance for reduced-rank model (per response)
        eta_r = X @ (B @ V[r, :].T) + m[r]
        pi_r = 1 / (1 + np.exp(-eta_r))
        pi_r = np.clip(pi_r, 1e-8, 1 - 1e-8)
        logL_r = np.sum(y * np.log(pi_r) + (1 - y) * np.log(1 - pi_r))
        D_rankS_r = -2 * logL_r
        
        # 4️⃣ Compute Qr
        Qr = (D0 - D_rankS_r) / (D0 - D_full)
        Qr_values.append(Qr)
    
    return np.array(Qr_values)


In [15]:
# ===============================================================
# SECTION: Timing Comparison — MM Algorithm vs IRLS Baseline
# ===============================================================

import time
import numpy as np
from sklearn.linear_model import LogisticRegression

# ---------------------------------------------------------------
# 1️⃣ Define the IRLS Baseline (unconstrained full logistic model)
# ---------------------------------------------------------------
def fit_IRLS_baseline(X, Y):
    """Fit R separate logistic regressions (unconstrained baseline)."""
    R = Y.shape[1]
    models = []
    for r in range(R):
        lr = LogisticRegression(max_iter=300, solver='lbfgs')
        lr.fit(X, Y[:, r])
        models.append(lr)
    return models


# ---------------------------------------------------------------
# 2️⃣ Define the Benchmark Comparison Function
# ---------------------------------------------------------------
def benchmark_algorithms(X, Y, B, V, m, S, n_iter=5):
    mm_times, irls_times = [], []
    
    for _ in range(n_iter):
        # Benchmark MM algorithm
        start = time.time()
        _ = logistic_rrr_MM(X, Y, S=S, maxiter=50, tol=1e-5)
        mm_times.append(time.time() - start)

        # Benchmark IRLS baseline
        start = time.time()
        _ = fit_IRLS_baseline(X, Y)
        irls_times.append(time.time() - start)
    
    return np.mean(mm_times), np.mean(irls_times)


# ---------------------------------------------------------------
# 3️⃣ Fit MM Model Once (Initialize Parameters)
# ---------------------------------------------------------------
# Make sure logistic_rrr_MM is already defined from earlier cells
B, V, m = logistic_rrr_MM(X, Y, S=1, maxiter=30, tol=1e-5)

# ---------------------------------------------------------------
# 4️⃣ Run Benchmark and Print Results
# ---------------------------------------------------------------
mm_t, irls_t = benchmark_algorithms(X, Y, B, V, m, S=1)

print(f"MM algorithm time: {mm_t:.4f}s | IRLS baseline: {irls_t:.4f}s")
print(f"Speed-up factor ≈ {irls_t/mm_t:.1f}×")


C:\Users\abhis\AppData\Local\Temp\ipykernel_17516\567730508.py:49: RuntimeWarning: divide by zero encountered in divide
  inv_sqrt = eigvecs @ np.diag(1/np.sqrt(eigvals)) @ eigvecs.T
C:\Users\abhis\AppData\Local\Temp\ipykernel_17516\567730508.py:49: RuntimeWarning: invalid value encountered in matmul
  inv_sqrt = eigvecs @ np.diag(1/np.sqrt(eigvals)) @ eigvecs.T


AttributeError: 'Series' object has no attribute 'reshape'

In [11]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def visualize_timing(mm_time, irls_time, dataset_name="Dataset"):
    # Prepare dataframe
    df_time = pd.DataFrame({
        "Algorithm": ["MM (Proposed)", "IRLS (Baseline)"],
        "Time (s)": [mm_time, irls_time]
    })
    df_time["Dataset"] = dataset_name
    
    # Plot
    plt.figure(figsize=(6,4))
    sns.barplot(data=df_time, x="Dataset", y="Time (s)", hue="Algorithm", palette="viridis")
    plt.title(f"Algorithm Timing Comparison — {dataset_name}", fontsize=13)
    plt.ylabel("Execution Time (s)")
    plt.xlabel("")
    plt.legend(title="Algorithm", loc="upper right")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

# Example visualization using benchmark results
visualize_timing(mm_t, irls_t, dataset_name="UCI Drug Consumption")


NameError: name 'mm_t' is not defined

In [ ]:
visualize_timing(mm_t_drug, irls_t_drug, dataset_name="UCI Drug Data")
visualize_timing(mm_t_ncd, irls_t_ncd, dataset_name="SHARE NCD Data")


In [ ]:
def get_triplot_coords(X, B, V, m):
    """
    Compute coordinates for Type I, Type D, and Hybrid triplots.
    Returns object scores, predictor axes, and response category points.
    """
    # Object coordinates (scores)
    U = X @ B  # N x 2 for rank=2
    
    # Predictor axes (directions)
    pred_axes = pd.DataFrame(B, columns=['Dim1', 'Dim2'])
    
    # Response variable axes
    resp_axes = pd.DataFrame(V, columns=['Dim1', 'Dim2'])
    
    # Compute Type D coordinates for "yes" and "no" categories
    resp_yes, resp_no = [], []
    for r in range(V.shape[0]):
        v_r = V[r, :]
        m_r = m[r]
        v_norm2 = np.dot(v_r, v_r)
        w_r0 = v_r * (-m_r/v_norm2 - 0.5)
        w_r1 = v_r * (-m_r/v_norm2 + 0.5)
        resp_no.append(w_r0)
        resp_yes.append(w_r1)
    
    resp_yes = np.array(resp_yes)
    resp_no = np.array(resp_no)
    
    return U, pred_axes, resp_axes, resp_yes, resp_no


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_typeI(U, pred_axes, resp_axes, resp_labels=None, pred_labels=None):
    plt.figure(figsize=(7,7))
    
    # Objects (participants)
    plt.scatter(U[:,0], U[:,1], color='gray', alpha=0.5, s=20, label='Participants')
    
    # Predictor axes
    for i in range(len(pred_axes)):
        plt.arrow(0, 0, pred_axes.iloc[i,0]*1.5, pred_axes.iloc[i,1]*1.5,
                  color='blue', alpha=0.7, width=0.01)
        if pred_labels is not None:
            plt.text(pred_axes.iloc[i,0]*1.6, pred_axes.iloc[i,1]*1.6, pred_labels[i], color='blue')
    
    # Response axes
    for j in range(len(resp_axes)):
        plt.arrow(0, 0, resp_axes.iloc[j,0]*1.5, resp_axes.iloc[j,1]*1.5,
                  color='green', alpha=0.7, width=0.01)
        if resp_labels is not None:
            plt.text(resp_axes.iloc[j,0]*1.6, resp_axes.iloc[j,1]*1.6, resp_labels[j], color='green')
    
    plt.axhline(0, color='black', lw=0.5)
    plt.axvline(0, color='black', lw=0.5)
    plt.title("Type I Triplot: Inner-Product Geometry", fontsize=13)
    plt.xlabel("Dimension 1"); plt.ylabel("Dimension 2")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


In [ ]:
def plot_typeD(U, pred_axes, resp_yes, resp_no, resp_labels=None):
    plt.figure(figsize=(7,7))
    plt.scatter(U[:,0], U[:,1], color='gray', alpha=0.5, label='Participants')
    
    # Predictor axes (same as Type I)
    for i in range(len(pred_axes)):
        plt.arrow(0, 0, pred_axes.iloc[i,0]*1.5, pred_axes.iloc[i,1]*1.5,
                  color='blue', alpha=0.6, width=0.01)
    
    # Response categories
    for r in range(resp_yes.shape[0]):
        plt.plot([resp_no[r,0], resp_yes[r,0]], [resp_no[r,1], resp_yes[r,1]],
                 color='green', lw=2)
        plt.scatter(resp_no[r,0], resp_no[r,1], color='red', marker='x', s=50)
        plt.scatter(resp_yes[r,0], resp_yes[r,1], color='green', marker='o', s=50)
        if resp_labels is not None:
            plt.text(resp_yes[r,0]*1.1, resp_yes[r,1]*1.1, resp_labels[r], color='darkgreen')
    
    plt.axhline(0, color='black', lw=0.5)
    plt.axvline(0, color='black', lw=0.5)
    plt.title("Type D Triplot: Distance Geometry", fontsize=13)
    plt.xlabel("Dimension 1"); plt.ylabel("Dimension 2")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


In [ ]:
def plot_hybrid(U, pred_axes, resp_axes, resp_yes, resp_no, resp_labels=None, pred_labels=None):
    plt.figure(figsize=(7,7))
    
    # Objects
    plt.scatter(U[:,0], U[:,1], color='lightgray', alpha=0.6, s=20)
    
    # Predictor axes
    for i in range(len(pred_axes)):
        plt.arrow(0, 0, pred_axes.iloc[i,0]*1.5, pred_axes.iloc[i,1]*1.5,
                  color='blue', alpha=0.7, width=0.01)
        if pred_labels is not None:
            plt.text(pred_axes.iloc[i,0]*1.6, pred_axes.iloc[i,1]*1.6, pred_labels[i], color='blue')
    
    # Response lines (wr0–wr1)
    for r in range(resp_yes.shape[0]):
        plt.plot([resp_no[r,0], resp_yes[r,0]], [resp_no[r,1], resp_yes[r,1]],
                 color='green', lw=2)
        plt.scatter(resp_yes[r,0], resp_yes[r,1], color='green', marker='o', s=50)
        plt.scatter(resp_no[r,0], resp_no[r,1], color='red', marker='x', s=50)
        if resp_labels is not None:
            plt.text(resp_yes[r,0]*1.1, resp_yes[r,1]*1.1, resp_labels[r], color='darkgreen')
    
    plt.axhline(0, color='black', lw=0.5)
    plt.axvline(0, color='black', lw=0.5)
    plt.title("Hybrid Triplot (Type I + Type D Combined)", fontsize=13)
    plt.xlabel("Dimension 1"); plt.ylabel("Dimension 2")
    plt.grid(alpha=0.3)
    plt.show()


In [ ]:
U, pred_axes, resp_axes, resp_yes, resp_no = get_triplot_coords(X, B, V, m)

pred_labels = X.columns if hasattr(X, "columns") else [f"P{i+1}" for i in range(B.shape[0])]
resp_labels = [f"Y{j+1}" for j in range(V.shape[0])]

plot_typeI(U, pred_axes, resp_axes, resp_labels, pred_labels)
plot_typeD(U, pred_axes, resp_yes, resp_no, resp_labels)
plot_hybrid(U, pred_axes, resp_axes, resp_yes, resp_no, resp_labels, pred_labels)


In [ ]:
# ============================================================
# SECTION: Synthetic Example — Type I, Type D & Hybrid Triplots
# ============================================================

# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------------------------------------------
# STEP 1: Generate synthetic data
# ------------------------------------------------------------

np.random.seed(42)
N, P, R, S = 150, 4, 3, 2  # 150 obs, 4 predictors, 3 responses, rank=2

# Predictors: X ~ N(0,1)
X = np.random.randn(N, P)
X = (X - X.mean(axis=0)) / X.std(axis=0)  # standardize

# True low-rank structure
B_true = np.random.randn(P, S)
V_true = np.random.randn(R, S)
m_true = np.random.randn(R)

# Logistic probabilities
eta = X @ (B_true @ V_true.T) + np.ones((N,1)) @ m_true.reshape(1,R)
pi = 1 / (1 + np.exp(-eta))

# Responses: Bernoulli draws
Y = (np.random.rand(N, R) < pi).astype(int)

# ------------------------------------------------------------
# STEP 2: Fit a rank-2 model using our MM function
# (we reuse the function logistic_rrr_MM from before)
# ------------------------------------------------------------

# MM algorithm implementation (short version)
def logistic_rrr_MM(X, Y, S, maxiter=100, tol=1e-5):
    N, P = X.shape
    R = Y.shape[1]
    B = np.random.randn(P, S)
    V = np.random.randn(R, S)
    m = np.log(Y.mean(axis=0) / (1 - Y.mean(axis=0)))  # initialize with marginal logits

    XtX = X.T @ X
    eigvals, eigvecs = np.linalg.eigh(XtX)
    inv_sqrt = eigvecs @ np.diag(1/np.sqrt(eigvals)) @ eigvecs.T
    sqrt_XtX = eigvecs @ np.diag(np.sqrt(eigvals)) @ eigvecs.T

    prev_ll = -np.inf
    for it in range(maxiter):
        # Linear predictor & probabilities
        eta = X @ (B @ V.T) + np.ones((N,1)) @ m.reshape(1,R)
        pi = 1 / (1 + np.exp(-eta))
        Z = eta + 4*(Y - pi)

        # Update intercepts
        D = Z - X @ (B @ V.T)
        m = D.mean(axis=0)
        U = Z - np.ones((N,1)) @ m.reshape(1,R)

        # GSVD update
        M = inv_sqrt @ (X.T @ U)
        Pmat, svals, Qt = np.linalg.svd(M, full_matrices=False)
        P_S = Pmat[:, :S]
        Q_S = Qt[:S, :].T
        B = np.sqrt(N) * inv_sqrt @ P_S
        V = (1/np.sqrt(N)) * sqrt_XtX @ Q_S

        # Log-likelihood check
        eta = X @ (B @ V.T) + np.ones((N,1)) @ m.reshape(1,R)
        pi = 1/(1+np.exp(-eta))
        ll = np.sum(Y*np.log(pi+1e-8) + (1-Y)*np.log(1-pi+1e-8))
        if np.abs(ll - prev_ll) < tol:
            break
        prev_ll = ll
    return B, V, m

# Fit the model
B, V, m = logistic_rrr_MM(X, Y, S=2)

# ------------------------------------------------------------
# STEP 3: Compute coordinates for triplots
# ------------------------------------------------------------

def get_triplot_coords(X, B, V, m):
    U = X @ B  # object coordinates
    pred_axes = pd.DataFrame(B, columns=['Dim1','Dim2'])
    resp_axes = pd.DataFrame(V, columns=['Dim1','Dim2'])
    resp_yes, resp_no = [], []
    for r in range(V.shape[0]):
        v_r = V[r, :]
        m_r = m[r]
        v_norm2 = np.dot(v_r, v_r)
        w_r0 = v_r * (-m_r/v_norm2 - 0.5)
        w_r1 = v_r * (-m_r/v_norm2 + 0.5)
        resp_no.append(w_r0)
        resp_yes.append(w_r1)
    return U, pred_axes, resp_axes, np.array(resp_yes), np.array(resp_no)

U, pred_axes, resp_axes, resp_yes, resp_no = get_triplot_coords(X, B, V, m)
pred_labels = [f"P{i+1}" for i in range(P)]
resp_labels = [f"Y{j+1}" for j in range(R)]

# ------------------------------------------------------------
# STEP 4: Plot Type I Triplot (Inner Product Geometry)
# ------------------------------------------------------------

plt.figure(figsize=(7,7))
plt.scatter(U[:,0], U[:,1], c='gray', alpha=0.5, s=20, label='Objects')
for i in range(P):
    plt.arrow(0,0,pred_axes.iloc[i,0]*1.5,pred_axes.iloc[i,1]*1.5,
              color='blue',alpha=0.6,width=0.01)
    plt.text(pred_axes.iloc[i,0]*1.6,pred_axes.iloc[i,1]*1.6,pred_labels[i],color='blue')
for r in range(R):
    plt.arrow(0,0,resp_axes.iloc[r,0]*1.5,resp_axes.iloc[r,1]*1.5,
              color='green',alpha=0.6,width=0.01)
    plt.text(resp_axes.iloc[r,0]*1.6,resp_axes.iloc[r,1]*1.6,resp_labels[r],color='green')
plt.title("Type I Triplot: Inner-Product Geometry")
plt.axhline(0,color='black',lw=0.5); plt.axvline(0,color='black',lw=0.5)
plt.grid(alpha=0.3); plt.legend(); plt.show()

# ------------------------------------------------------------
# STEP 5: Plot Type D Triplot (Distance Geometry)
# ------------------------------------------------------------

plt.figure(figsize=(7,7))
plt.scatter(U[:,0], U[:,1], c='gray', alpha=0.5, s=20, label='Objects')
for i in range(P):
    plt.arrow(0,0,pred_axes.iloc[i,0]*1.5,pred_axes.iloc[i,1]*1.5,
              color='blue',alpha=0.6,width=0.01)
for r in range(R):
    plt.plot([resp_no[r,0], resp_yes[r,0]],[resp_no[r,1], resp_yes[r,1]],color='green',lw=2)
    plt.scatter(resp_no[r,0], resp_no[r,1],color='red',marker='x',s=50)
    plt.scatter(resp_yes[r,0], resp_yes[r,1],color='green',marker='o',s=50)
    plt.text(resp_yes[r,0]*1.1,resp_yes[r,1]*1.1,resp_labels[r],color='darkgreen')
plt.title("Type D Triplot: Distance Geometry")
plt.axhline(0,color='black',lw=0.5); plt.axvline(0,color='black',lw=0.5)
plt.grid(alpha=0.3); plt.legend(); plt.show()

# ------------------------------------------------------------
# STEP 6: Plot Hybrid Triplot (Combined View)
# ------------------------------------------------------------

plt.figure(figsize=(7,7))
plt.scatter(U[:,0],U[:,1],c='lightgray',alpha=0.6,s=20,label='Objects')
for i in range(P):
    plt.arrow(0,0,pred_axes.iloc[i,0]*1.5,pred_axes.iloc[i,1]*1.5,
              color='blue',alpha=0.7,width=0.01)
    plt.text(pred_axes.iloc[i,0]*1.6,pred_axes.iloc[i,1]*1.6,pred_labels[i],color='blue')
for r in range(R):
    plt.plot([resp_no[r,0],resp_yes[r,0]],[resp_no[r,1],resp_yes[r,1]],color='green',lw=2)
    plt.scatter(resp_no[r,0],resp_no[r,1],color='red',marker='x',s=50)
    plt.scatter(resp_yes[r,0],resp_yes[r,1],color='green',marker='o',s=50)
    plt.text(resp_yes[r,0]*1.1,resp_yes[r,1]*1.1,resp_labels[r],color='darkgreen')
plt.title("Hybrid Triplot (Type I + Type D Combined)")
plt.axhline(0,color='black',lw=0.5); plt.axvline(0,color='black',lw=0.5)
plt.grid(alpha=0.3); plt.legend(); plt.show()
